In [ ]:
# imports necesarios
import pandas as pd 
import os

In [ ]:
# definimos el directorio de trabajo
path = os.chdir(r'C:\PROYECTOS _VSC\tendencias_consumo')
os.getcwd()

# Carga de Datos

In [ ]:
df = pd.read_csv("ventas-totales-supermercados-2.csv")

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
# Se convierte el tipo de dato de objetc a datetime
df['indice_tiempo'] = pd.to_datetime(df['indice_tiempo'])

In [ ]:
df['indice_tiempo'].dtype

## Filtramos las columnas de interés

In [ ]:
columnas_interes = ['indice_tiempo','salon_ventas','canales_on_line']
df_modelo = df[columnas_interes]

In [ ]:
df_modelo.head()

# Búsqueda de anomalías

In [ ]:
# Comparamos la media vs la mediana para detectar si existen valores extremos (Outliers)
resumen_estadistico = df_modelo[['salon_ventas', 'canales_on_line']].agg(['mean', 'median'])
# Mostramos el resultado
print(resumen_estadistico)

In [ ]:
# Dividimos por 1_000_000 para que el gráfico muestre "Millones" y sea legible
ax = (df_modelo[['salon_ventas', 'canales_on_line']] / 1_000_000).plot.box()
ax.set_ylabel('Ventas (millones)')

In [ ]:
# Al usar subplots=True, cada canal de venta tendrá su propia escala vertical
(df_modelo[['salon_ventas', 'canales_on_line']] / 1_000_000).plot.box(subplots=True, figsize=(10, 5))

### Conclusión del Análisis de Anomalías:

Al comparar las métricas de localización (media vs. mediana) y generar los diagramas de caja, se confirma la presencia de múltiples valores atípicos superiores en ambos canales de venta (Salón y Online). Dado que la Regresión Lineal es muy sensible a valores extremos, estos registros (posiblemente causados por la inflación reciente) deberán tenerse en cuenta al momento de evaluar el rendimiento del modelo.

# Análisis de Correlación

In [ ]:
# 1. Matriz de Correlación: Medida matemática (va de -1 a +1)
correlacion = df_modelo[['salon_ventas', 'canales_on_line']].corr()
print("Matriz de Correlación:")
print(correlacion)

In [ ]:
# 2. Diagrama de dispersión (Scatter plot): Visualización de la relación
ax = df_modelo.plot.scatter(x='salon_ventas', y='canales_on_line', figsize=(6, 4))
ax.set_xlabel('Ventas en Salón')
ax.set_ylabel('Ventas Online')

Los datos demuestran matemáticamente que cuando las ventas en el supermercado físico suben, las ventas por internet suben en una proporción casi exacta, moviéndose juntas.

# Detección de Multicolinealidad

Condición en la que hay redundancia entre las variables predictoras. Se da cuando múltiples variables "están casi perfectamente correlacionadas entre sí" y puede causar inestabilidad numérica al ajustar la ecuación de regresión.

**Solución:** Las variables deben eliminarse hasta que desaparezca la multicolinealidad.

**Conclusión:** No se realizará una Regresión Lineal Múltiple, sino que se aplicará el modelo más estable para este caso: una Regresión Lineal Simple. Se utilizará una única variable predictora fuerte (salon_ventas) para para predecir la variable de respuesta o resultado (canales_on_line).


In [ ]:
# 1. Seleccionamos posibles variables predictoras de tu dataset original
columnas_predictoras = ['salon_ventas', 'efectivo', 'tarjetas_debito', 'tarjetas_credito']
df_predictores = df[columnas_predictoras]
# 2. Generamos la matriz de correlación para detectar Multicolinealidad
matriz_multicolinealidad = df_predictores.corr()

print("Matriz de Multicolinealidad entre Predictores:")
print(matriz_multicolinealidad)

## Decisión metodológica

Para resolver el problema del impacto inflacionario y la presencia de valores atípicos (outliers) en los meses recientes, se toma la decisión de no predecir valores monetarios nominales. En su lugar, se calculará el porcentaje de ventas online sobre el total de ventas. Al trabajar con proporciones, el sesgo de la inflación se cancela matemáticamente, lo que estabilizará la Regresión Lineal sin necesidad de eliminar datos históricos válidos.

In [ ]:
# Creamos una copia limpia para nuestro modelo definitivo
df_final = df_modelo.copy()

In [ ]:
# Calculamos las ventas totales del mes
ventas_totales = df_final['salon_ventas'] + df_final['canales_on_line']

In [ ]:
# Creamos la nueva variable: Porcentaje de ventas online
df_final['porcentaje_online'] = (df_final['canales_on_line'] / ventas_totales) * 100


In [ ]:
# Creamos la columna numérica (1, 2, 3...)
df_final['mes_numero'] = range(1, len(df_final) + 1)

In [ ]:
# Visualizamos las primeras filas para confirmar
df_final.head()

In [ ]:
# Visualizamos las últimas filas 
df_final.tail()

# Regresión Lineal Simple

### Uso del 100% de los datos en Regresión Lineal

Se entrena el modelo con todas las observaciones disponibles, sin dividir el dataset en entrenamiento y prueba. Dado que la Regresión Lineal Simple presenta bajo riesgo de sobreajuste, se priorizó aprovechar toda la información histórica para estimar la tendencia global de la manera más precisa posible y brindar una referencia sólida para la toma de decisiones.

### Limitación Geográfica

Los datos están agregados a nivel nacional y no incluyen información de ubicación. Por ello, el modelo estima una tendencia promedio para toda Argentina. Esta simplificación puede ocultar diferencias regionales, ya que la adopción de canales online suele variar entre zonas urbanas y rurales.

In [ ]:
# Importamos el modelo de Regresión Lineal
from sklearn.linear_model import LinearRegression

# 1. Definimos el predictor (X) y el resultado que queremos predecir (y)
X = df_final[['mes_numero']]  # Debe mantenerse como DataFrame
y = df_final['porcentaje_online']

# 2. Inicializamos y entrenamos el modelo con el 100% de los datos
modelo = LinearRegression()
modelo.fit(X, y)

# 3. Extraemos los parámetros de la recta ajustada
print(f'Intersección (b0): {modelo.intercept_:.3f}')
print(f'Pendiente (b1): {modelo.coef_[0]:.3f}')

# 4. Mostramos la ecuación del modelo
print(f'Ecuación: y = {modelo.intercept_:.3f} + {modelo.coef_[0]:.3f}x')

### Interpretación de los Coeficientes

**Intersección (b₀ = 1.637):** Representa el valor estimado del porcentaje de ventas online cuando `mes_numero = 0`. En términos prácticos, indica que al inicio del período analizado la participación del canal online era aproximadamente del **1.637%** de las ventas totales.

**Pendiente (b₁ = 0.020):** Es el crecimiento promedio mensual del porcentaje de ventas online. Esto significa que la participación del canal digital aumenta, en promedio, **0.020 puntos porcentuales por mes**, evidenciando una tendencia sostenida de expansión del comercio electrónico a lo largo del tiempo.

### Predicción
**Si la tendencia se mantiene, ¿qué porcentaje de nuestra logística debe estar dedicada a envíos online para el cierre de año, en Diciembre de 2026**

Desde enero de 2017 hasta el último dato de febrero de 2026 hay exactamente 110 meses. Por lo tanto, marzo de 2026 será el mes 111... y diciembre de 2026 será el mes 120.
Aplicamos la ecuación: y=1.637+(0.020×120) y=1.637+2.400 y=4.037

De acuerdo con la tendencia histórica de los últimos 9 años, se estima que para diciembre de 2026, aproximadamente el 4.04% del volumen total de ventas provendrá del canal online.